# 02_feature_engineering.ipynb
Creación de nuevas variables a partir del dataset limpio.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
sys.path.insert(0, os.path.abspath('..'))
from src.feature_engineering import (
    extract_bhk, luxury_flag, furnishing_score, has_garden_view
 )

df = pd.read_csv('../data/processed/house_prices_clean.csv')

df['bhk_count'] = df['Title'].apply(extract_bhk).fillna(0).astype(int)
df['price_per_sqft'] = df.apply(
    lambda row: row['amount_rupees'] / row['carpet_area_num'] 
    if row['carpet_area_num'] and row['carpet_area_num'] > 0 else np.nan,
    axis=1
)
df['floor_ratio'] = df.apply(
    lambda row: row['floor_number'] / row['total_floors'] 
    if row['total_floors'] and row['total_floors'] > 0 else 0,
    axis=1
)
df['is_luxury'] = df['Title'].fillna('') + ' ' + df['Description'].fillna('')
df['is_luxury'] = df['is_luxury'].apply(luxury_flag)
df['furnishing_score'] = df['Furnishing'].apply(furnishing_score)
df['has_garden_view'] = df['overlooking'].apply(has_garden_view)
df['has_society'] = df['Society'].notna().astype(int)
df['city_avg_price'] = df.groupby('location')['amount_rupees'].transform('median')

df[['bhk_count','price_per_sqft','floor_ratio','is_luxury','furnishing_score','has_garden_view','city_avg_price','has_society']].head()

C:\Users\juang\AppData\Local\Temp\ipykernel_3224\72676105.py:10: DtypeWarning: Columns (0: Society) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/house_prices_clean.csv')


,bhk_count,price_per_sqft,floor_ratio,is_luxury,furnishing_score,has_garden_view,city_avg_price,has_society
0,1,8400.000000,0.909091,0,0,0,6800000.0,1
1,2,20718.816068,0.136364,1,1,1,6800000.0,1
2,2,17971.758665,0.344828,0,0,1,6800000.0,1
3,1,4716.981132,0.333333,0,0,0,6800000.0,0
4,2,25196.850394,0.476190,1,0,1,6800000.0,1


## Verificación de nuevas variables
Revisar distribución y valores nulos después de la ingeniería de atributos.

In [ ]:
print(df[['price_per_sqft', 'floor_ratio', 'is_luxury', 'furnishing_score', 'has_garden_view', 'city_avg_price']].describe())
print('Nulos por columna:')
print(df[['price_per_sqft', 'floor_ratio', 'is_luxury', 'furnishing_score', 'has_garden_view', 'city_avg_price']].isnull().mean().round(4) * 100)

       price_per_sqft    floor_ratio      is_luxury  furnishing_score  \
count   162606.000000  162606.000000  162606.000000     162606.000000   
mean     10317.459563       0.501650       0.425009          0.678314   
std      12258.743224       0.307331       0.494346          0.662514   
min         62.678063       0.000000       0.000000          0.000000   
25%       5000.000000       0.250000       0.000000          0.000000   
50%       7500.000000       0.500000       0.000000          1.000000   
75%      11200.000000       0.750000       1.000000          1.000000   
max     294117.647059       3.000000       1.000000          2.000000   

       has_garden_view  city_avg_price  
count    162606.000000    1.626060e+05  
mean          0.365411    8.396115e+06  
std           0.481547    4.041902e+06  
min           0.000000    2.210000e+06  
25%           0.000000    5.500000e+06  
50%           0.000000    6.800000e+06  
75%           1.000000    8.500000e+06  
max           

## Guardar dataset con variables nuevas
Este archivo será usado en modelado y clustering.

In [ ]:
df.to_csv('../data/processed/house_prices_clean.csv', index=False)
print('Dataset con nuevas variables guardado en data/processed/house_prices_clean.csv')

Dataset con nuevas variables guardado en data/processed/house_prices_clean.csv


## Visualizaciones Exploratorias
Genera y guarda las 5 gráficas requeridas en reports/figures/

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np

figures_path = os.path.abspath(os.path.join("..", "reports", "figures"))
os.makedirs(figures_path, exist_ok=True)


### VIZ 1 — Distribución del precio (histograma + KDE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["amount_rupees"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribución del Precio (original)")
axes[0].set_xlabel("Precio (Rupias)")

sns.histplot(np.log1p(df["amount_rupees"]), kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribución del Precio (log-transform)")
axes[1].set_xlabel("log(Precio + 1)")

plt.tight_layout()
plt.savefig(os.path.join(figures_path, "01_distribucion_precio.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Sesgo confirmado: distribución original sesgada a la derecha.")
print("Log-transform justificado para modelos lineales (Ridge).")


### VIZ 2 — Heatmap de correlación

In [ ]:
numeric_cols = [
    "amount_rupees", "carpet_area_num", "Bathroom", "Balcony",
    "bhk_count", "floor_number", "total_floors", "floor_ratio",
    "furnishing_score", "is_luxury", "has_parking", "city_avg_price"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, square=True)
plt.title("Matriz de Correlación — Variables Numéricas")
plt.tight_layout()
plt.savefig(os.path.join(figures_path, "02_heatmap_correlacion.png"), dpi=150, bbox_inches="tight")
plt.show()

top3 = corr["amount_rupees"].drop("amount_rupees").abs().sort_values(ascending=False).head(3)
print("Top 3 variables más correlacionadas con el precio:")
print(top3)


### VIZ 3 — Boxplots de precio por categoría

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, ["Furnishing", "Transaction", "bhk_count"]):
    top_cats = df[col].value_counts().index[:6]
    sns.boxplot(
        data=df[df[col].isin(top_cats)],
        x=col, y="amount_rupees", ax=ax, palette="Set2"
    )
    ax.set_title(f"Precio por {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Precio (Rupias)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(figures_path, "03_boxplots_categorias.png"), dpi=150, bbox_inches="tight")
plt.show()


### VIZ 4 — Scatter interactivo Área vs Precio (Plotly)

In [ ]:
sample_df = df.sample(5000, random_state=42)

fig = px.scatter(
    sample_df,
    x="carpet_area_num",
    y="amount_rupees",
    color="bhk_count",
    hover_data=["location", "Furnishing", "floor_number"],
    title="Área vs Precio por Número de Habitaciones (BHK)",
    labels={
        "carpet_area_num": "Área útil (sqft)",
        "amount_rupees": "Precio (Rupias)",
        "bhk_count": "BHK"
    },
    opacity=0.6,
    color_continuous_scale="Viridis"
)
fig.write_html(os.path.join(figures_path, "04_scatter_interactivo.html"))
fig.show()
print("Guardado en reports/figures/04_scatter_interactivo.html")


### VIZ 5 — Precio mediano por ciudad (Top 15)

In [ ]:
city_price = (
    df.groupby("location")["amount_rupees"]
    .median()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(14, 6))
sns.barplot(x=city_price.index, y=city_price.values, palette="viridis")
plt.title("Precio Mediano por Ciudad (Top 15)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Precio mediano (Rupias)")
plt.xlabel("Ciudad")
plt.tight_layout()
plt.savefig(os.path.join(figures_path, "05_precio_por_ciudad.png"), dpi=150, bbox_inches="tight")
plt.show()

print("
Top 5 ciudades más caras:")
print(city_price.head())
